# Validación de `src/espectral.py` — Etapa 4

Este notebook valida el Modelo 6 (corte espectral con linealización de Taylor del residuo), usando el Modelo 4 solo como función auxiliar interna para el punto de expansión `x_bar`. Se reutiliza la misma ventana y $\mu$ ya validados en etapas anteriores.

In [10]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from src.config import cargar_config
from src import datos, estimacion, optimizacion, espectral

config = cargar_config()
factor_anual = config["estimacion"]["factor_anualizacion"]

## Preparación de datos

In [11]:
precios = datos.obtener_precios(config)

periodo_inicio = "2025-03-11"
periodo_fin = "2025-09-18"

ventana = precios.loc[periodo_inicio:periodo_fin]
ventana_filtrada = datos.eliminar_columnas_incompletas(ventana, umbral=0.10)
ventana_imputada = datos.imputar_precios(ventana_filtrada)
rendimientos_log = datos.calcular_rendimientos_log(ventana_imputada)

mu = estimacion.estimar_mu_bayes_stein(rendimientos_log, factor_anual=factor_anual)
Sigma_muestral = estimacion.estimar_sigma_muestral(rendimientos_log, factor_anual=factor_anual)
Sigma_lw = estimacion.estimar_sigma_ledoit_wolf(rendimientos_log, factor_anual=factor_anual)

mu_P = 0.10

print(f"Shape rendimientos: {rendimientos_log.shape}")
print(f"Retorno objetivo (mu_P): {mu_P}")

Shape rendimientos: (132, 110)
Retorno objetivo (mu_P): 0.1


## Descomposición espectral: $\Sigma$ muestral vs. Ledoit-Wolf

En la Etapa 2 verificamos que Ledoit-Wolf mejora dramáticamente el número de condición de $\Sigma$ (de `2.31e+19` a `16.5`). Eso es excelente para invertir la matriz (en la Etapa 3), pero tiene una consecuencia directa sobre el corte espectral: Ledoit-Wolf encoge los eigenvalores pequeños hacia un piso común, aplanando el espectro. El Modelo 6 depende de que los eigenvalores descartados sean genuinamente pequeños (cercanos a cero) para que el residuo linealizado sea una buena aproximación.

In [12]:
eigenvalues_m, eigenvectors_m = espectral.descomposicion_espectral(Sigma_muestral)
eigenvalues_lw, eigenvectors_lw = espectral.descomposicion_espectral(Sigma_lw)

k90_m = espectral.num_componentes_umbral(eigenvalues_m, umbral=0.90)
k90_lw = espectral.num_componentes_umbral(eigenvalues_lw, umbral=0.90)

print(f"k (90% varianza) con Sigma muestral: {k90_m} / {len(mu)}")
print(f"k (90% varianza) con Sigma Ledoit-Wolf: {k90_lw} / {len(mu)}")
print(f"Eigenvalores muestral (min, max): ({eigenvalues_m.min():.2e}, {eigenvalues_m.max():.2e})")
print(f"Eigenvalores Ledoit-Wolf (min, max): ({eigenvalues_lw.min():.2e}, {eigenvalues_lw.max():.2e})")

k (90% varianza) con Sigma muestral: 40 / 110
k (90% varianza) con Sigma Ledoit-Wolf: 87 / 110
Eigenvalores muestral (min, max): (-2.77e-17, 1.31e+00)
Eigenvalores Ledoit-Wolf (min, max): (4.56e-02, 7.53e-01)


## Validación teórica de la cota superior: $V^* \leq$ cota superior

La única garantía teórica estricta es $V^* \leq$ cota superior: los pesos del Modelo 6 son un portafolio *factible* (cumple las mismas restricciones que el problema exacto), y $V^*$ es el mínimo global sobre todo el conjunto factible — por definición, ningún portafolio factible puede tener varianza real menor a $V^*$.

La cota inferior (el valor del objetivo linealizado de Modelo 6 en su propio óptimo) no tiene esa misma garantía frente a $V^*$: a diferencia de $V_I^*$ del Modelo 4 (que sí es una relajación estricta, por ignorar por completo la varianza de los componentes descartados), el objetivo de Modelo 6 combina el término exacto de los primeros $I$ componentes con una corrección de Taylor linealizada (con recorte `max(·, 0)`) del residuo — una aproximación, no una relajación garantizada. Por eso reportamos la cota inferior como referencia informativa, y solo exigimos la desigualdad $V^* \leq$ cota superior.

In [13]:
for I in [5, 20, 40, 60, 93, 109]:
    cotas = espectral.cotas_modelo_6(mu, Sigma_muestral, mu_P, I)
    cumple = cotas["V_optimo"] <= cotas["cota_superior"] + 1e-12
    print(f"I={I:>3}: cota_inf={cotas['cota_inferior']:.6e}  V*={cotas['V_optimo']:.6e}  "
          f"cota_sup={cotas['cota_superior']:.6e}  V*<=cota_sup={cumple}  error={cotas['error_relativo_pct']:.2f}%")
    assert cumple

I=  5: cota_inf=1.086843e-07  V*=2.186110e-05  cota_sup=4.647488e-04  V*<=cota_sup=True  error=2025.92%
I= 20: cota_inf=2.613871e-07  V*=2.186110e-05  cota_sup=2.700896e-04  V*<=cota_sup=True  error=1135.48%
I= 40: cota_inf=9.191012e-06  V*=2.186110e-05  cota_sup=1.627738e-04  V*<=cota_sup=True  error=644.58%
I= 60: cota_inf=1.390293e-06  V*=2.186110e-05  cota_sup=1.116403e-04  V*<=cota_sup=True  error=410.68%
I= 93: cota_inf=4.826321e-05  V*=2.186110e-05  cota_sup=4.826321e-05  V*<=cota_sup=True  error=120.77%
I=109: cota_inf=5.282532e-05  V*=2.186110e-05  cota_sup=5.282532e-05  V*<=cota_sup=True  error=141.64%


## `x_bar` explícito vs. calculado internamente (Modelo 4 como auxiliar)

`modelo_6` debe dar exactamente el mismo resultado si se le pasa `x_bar` ya calculado, o si lo calcula internamente llamando a `_modelo_4` (comportamiento por default, `x_bar=None`).

In [14]:
I_prueba = k90_m
x_bar_manual, _ = espectral._modelo_4(mu.values, mu_P, eigenvalues_m, eigenvectors_m, I_prueba)

pesos_auto, m_auto = espectral.modelo_6(mu, mu_P, eigenvalues_m, eigenvectors_m, I_prueba, Sigma_muestral)
pesos_manual, m_manual = espectral.modelo_6(mu, mu_P, eigenvalues_m, eigenvectors_m, I_prueba, Sigma_muestral, x_bar=x_bar_manual)

diff = np.max(np.abs(pesos_auto.values - pesos_manual.values))
print(f"Diferencia maxima entre x_bar automatico y manual: {diff:.2e}")
assert diff < 1e-10

Diferencia maxima entre x_bar automatico y manual: 0.00e+00


## Convergencia del Modelo 6

Reproducimos la gráfica de convergencia del primer borrador: error relativo de la cota superior vs. número de componentes retenidos. Usamos un barrido completo ($I=1..N$) con $\Sigma$ muestral, y un barrido más espaciado con $\Sigma$ Ledoit-Wolf para contrastar (el barrido completo con Ledoit-Wolf tarda lo mismo pero sabemos de antemano que no va a converger a cero, dado el piso en su espectro).

In [15]:
n = len(mu)

convergencia_muestral = espectral.convergencia_modelo_6(mu, Sigma_muestral, mu_P, rango_I=range(1, n + 1))
convergencia_lw = espectral.convergencia_modelo_6(mu, Sigma_lw, mu_P, rango_I=range(1, n + 1, 10))

print(f"Sigma muestral - error en I=1: {convergencia_muestral['error_relativo_pct'][0]:.2f}%, "
      f"error en I={n}: {convergencia_muestral['error_relativo_pct'][-1]:.2f}%, "
      f"error minimo: {convergencia_muestral['error_relativo_pct'].min():.2f}%")
print(f"Sigma Ledoit-Wolf - error en I=1: {convergencia_lw['error_relativo_pct'][0]:.2f}%, "
      f"error en I={convergencia_lw['rango_I'][-1]}: {convergencia_lw['error_relativo_pct'][-1]:.2f}%, "
      f"error minimo: {convergencia_lw['error_relativo_pct'].min():.2f}%")

Sigma muestral - error en I=1: 9990.10%, error en I=110: 141.64%, error minimo: 120.77%
Sigma Ledoit-Wolf - error en I=1: 87.85%, error en I=101: 52.90%, error minimo: 3.30%


In [16]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=convergencia_muestral["rango_I"], y=convergencia_muestral["error_relativo_pct"],
    mode="lines+markers", name="Sigma muestral", marker=dict(size=4)))
fig.add_trace(go.Scatter(
    x=convergencia_lw["rango_I"], y=convergencia_lw["error_relativo_pct"],
    mode="lines+markers", name="Sigma Ledoit-Wolf", marker=dict(size=6)))
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Convergencia del Modelo 6: error relativo de la cota superior vs. componentes retenidos",
    xaxis_title="Componentes retenidos (I)", yaxis_title="Error relativo (%)",
    template="plotly_white")
fig.show()

Con $\Sigma$ muestral el error decae varios órdenes de magnitud conforme $I$ se acerca al rango numérico de la matriz (los eigenvalores de la cola son ~1e-17, ruido de punto flotante). Con $\Sigma$ Ledoit-Wolf el error **no converge de forma confiable**: fluctúa erráticamente entre ~3% y ~3000% según $I$, sin la tendencia decreciente que sí se observa con $\Sigma$ muestral porque el shrinkage garantiza que ningún eigenvalor sea realmente pequeño (piso ≈ 0.046), así que agregar más componentes no reduce sistemáticamente el residuo.

El corte espectral (Modelo 6) es coherente solo si $\Sigma$ conserva una cola de eigenvalores genuinamente pequeña, típicamente $\Sigma$ muestral. Si el usuario elige Ledoit-Wolf en la interfaz para el método clásico, el corte espectral seguirá siendo matemáticamente válido (las cotas teóricas se cumplen) pero con errores de aproximación mucho mayores. Vale la pena mostrar `error_relativo_pct` como diagnóstico visible en la interfaz para que el usuario lo tenga en consideración.

## Frontera espectral

In [17]:
resultado_espectral = espectral.frontera_espectral(mu, Sigma_muestral, n_points=20, umbral_varianza=0.90)

print(f"Componentes usados (k): {resultado_espectral['k_componentes']}")
print(f"Sigma monotona creciente: {np.all(np.diff(resultado_espectral['sigma_eficiente']) >= -1e-8)}")
print(resultado_espectral["sigma_eficiente"][:5])

Componentes usados (k): 40
Sigma monotona creciente: True
[0.00960432 0.01156092 0.01649282 0.02267904 0.02966569]


## Salida uniforme para la interfaz: `optimizar_portafolio_espectral`

Debe producir el mismo formato que `optimizacion.resumen_portafolio` (pesos %, $\mu$, $\sigma$, Sharpe), más `k_componentes`.

In [18]:
resumen_espectral = espectral.optimizar_portafolio_espectral(mu, Sigma_muestral, mu_P=mu_P, r_f=0.02)

print(f"k_componentes: {resumen_espectral['k_componentes']}")
print(f"mu: {resumen_espectral['mu']:.4f}, sigma: {resumen_espectral['sigma']:.4f}, sharpe: {resumen_espectral['sharpe']:.4f}")
print(f"suma pesos_pct: {resumen_espectral['pesos_pct'].sum():.4f}")
display(resumen_espectral["pesos_pct"].head())

assert np.isclose(resumen_espectral["pesos_pct"].sum(), 100.0, atol=1e-3)

k_componentes: 40
mu: 0.1000, sigma: 0.0128, sharpe: 6.2704
suma pesos_pct: 100.0000


POCHTECB.MX    3.638223
ACTINVRB.MX    3.398723
FINAMEXO.MX    3.239131
ALTERNA.MX     3.183539
INVEXA.MX      3.053119
dtype: float64